In [6]:
development_flag : bool = True

In [4]:
#import module
import pandas as pd

READ CSV FILE


In [ ]:
#create dataframe
dataframe = pd.read_csv("archive/songs.csv")

SIMPLE DATA PREPROCESSING (remove newline chars, drop duplicates, remove short lyrics)


In [23]:
def  lyricPreprocess (frame : pd.DataFrame )  -> pd.DataFrame:
    # Remove newline chars
    frame['lyrics'] = frame['lyrics'].str.replace(r"\\n", " ", regex=True)
    frame['lyrics'] = frame['lyrics'].str.replace("\n", " ", regex=True)
    frame['lyrics'] = frame['lyrics'].str.replace("\r", " ", regex=True)
    #convert everything to lowercase
    frame['lyrics'] = frame['lyrics'].str.lower()
    #drop duplicates 
    frame['lyrics'] = frame['lyrics'].drop_duplicates()
    #remove short lyrics
    frame= frame[frame['lyrics'].notna() & (frame['lyrics'].str.len() > 250)]     
    
    return  frame[['lyrics','genre']]

SEE IF DATASET IS BALANCED (ONLY FOR DEVELOPMENT)


In [24]:
def getStatisticsOnDataset (frame : pd.DataFrame) -> pd.DataFrame : 
    #count duplicates based on lyrics 
    counts = frame['lyrics'].value_counts()
    counts = counts[counts > 1]
    print('Duplicates :' ,frame['lyrics'].duplicated().sum())
    genre_percent = frame['genre'].value_counts(normalize=True) * 100
    print(genre_percent)

SUBSET OF GENRES


In [1]:
selectedGenres : list[str] = ['Rock', 'Pop', 'Electronic', 'Folk']
selectGenres2  : list[str] = ['Rock', 'Jazz', 'Electronic','Hip-Hop']
selectGenres3  : list[str] = ['Rock', 'Jazz', 'R&B','Hip-Hop']
selectGenres4  : list[str] = ['Rock', 'Jazz', 'Classical', 'Hip-Hop']
selectGenres5  : list[str] = ['Classical', 'Country', 'Hip-Hop', 'Electronic']

In [ ]:
#choose the 2 subsets which the experiment is built on, one with higher f1-score,
# the other one with lower f1-score to see the model does not perform as good when the lyrics are semantically similar

sets : list[list[str]] = [selectGenres5, selectGenres2]

CREATE A BALANCED SUBSET


In [ ]:
def getBalancedSubset(frame : pd.DataFrame, subset :list[str])  -> pd.DataFrame:
    return frame[frame['genre'].isin(subset)].groupby('genre').sample(n=2500, random_state=69).reset_index(drop=True)

In [ ]:
def saveToDisk(frame : pd.DataFrame, subset : list[str]) :
    path = f'{"".join(subset)}/balancedSubset10k{"".join(subset)}.csv'
    frame.to_csv(path, index=True, encoding='utf-8')
     

PIPELINE

1. PREPROCESS LYRICS
2. CREATE BALANCED SUBSET
3. PRINT STATISTICS (OPTIONAL)
4. SAVE DATASET TO CSV FOR FURTHER TASK

In [ ]:
def preProcessPipeLine(frame : pd.DataFrame, sets : list): 
    #preprocess the whole dataset
    frame = lyricPreprocess(frame=frame)
    #iterate over both subsets needed for training
    for set in sets:
        #get a balanced dataset
        frame = getBalancedSubset(frame=frame,subset=set)
        #print statistics (optional)
        if development_flag : 
            getStatisticsOnDataset(frame=frame)
        #save dataset to disk (maybe leave this out?)
        saveToDisk(frame, set)

RUN THE PIPELINE

In [29]:
preProcessPipeLine(frame=dataframe)

Duplicates : 0
genre
Classical     25.0
Country       25.0
Electronic    25.0
Hip-Hop       25.0
Name: proportion, dtype: float64
